# BPI Challenge 2018: agricultural grant applications

Agricultural grant applications contain several document types, inspections, revisions and payment attempts. The business question is how to support valid initial payment within the funding-year target while protecting correct entitlement, inspection and objection handling.

The questions below concern the complete recorded case. WISE evaluates the declared constraints and returns the numerical summaries, priorities and evidence. Figures display those returned results. The review scales are exploratory inputs awaiting process-owner approval; a flagged case is a starting point for investigation.

## PM² route

The work follows the six stages of [PM²](https://vdaalst.com/publications/p824.pdf). Revisit earlier stages when evidence changes the question or its scope.

| Stage | Work recorded here | Decision required |
|---|---|---|
| 1. Planning | Business goal, case unit, interpretation questions and candidate norms | Confirm the investigation scope and acceptable service outcomes |
| 2. Extraction | Read the supplied challenge logs and identify source versions | Check the extraction boundary and missing records |
| 3. Data processing | Retain whole cases, review activity bindings and context | Resolve ambiguous meanings and episode boundaries |
| 4. Mining and analysis | Review severity, coverage, context, priorities and original traces | Identify a recurring mechanism supported by evidence |
| 5. Evaluation | Compare views, settings and plausible explanations | Decide whether a proposed change warrants a trial |
| 6. Process improvement and support | Assign an owner, define a pilot and protect essential work | Measure the result before expanding the change |

The execution journal records the analytical run. The review journal near the end records business judgements separately; running a notebook does not approve a norm or complete a process-improvement trial.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import wise

try:
    from bpic_analysis import (
        challenge, summary, norm_table, criterion_summary, activity_summary,
        context_profiles, priorities, plot_overview, case_evidence,
        threshold_sensitivity, view_agreement, integrity, save_outputs, journal,
    )
except ModuleNotFoundError as error:
    if error.name != "bpic_analysis":
        raise
    from notebooks.bpic_analysis import (
        challenge, summary, norm_table, criterion_summary, activity_summary,
        context_profiles, priorities, plot_overview, case_evidence,
        threshold_sensitivity, view_agreement, integrity, save_outputs, journal,
    )

if wise.__version__ != "1.0.0":
    raise RuntimeError("Select the environment with the frozen 1.0.0 package.")

YEAR = 2018
data_root = os.environ.get("WISE_PAPER_DATA")
SAVE_OUTPUTS = False


def show_evidence(evidence):
    for label, value in evidence.items():
        display(Markdown("**" + label.replace("_", " ") + "**"))
        display(value)


def show_figures(run):
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)

## 1. Planning: the business question

**Goal:** begin valid initial payments within the funding-year target and reduce confirmed avoidable readiness gaps or repeated internal corrections.

**Case unit:** the complete recorded application. Related documents and their versions remain part of its context. An applicant can have applications in different years; an applicant identifier is not the same unit as an application case.

Investigate:

1. Which funding-year and department groups concentrate aborts, revocations or revised-payment activity?
2. Which recorded patterns belong to initial applications, changes or objections, and which reflect necessary controls?
3. Can a reviewed payment attempt be linked to its document, decision version and any later cancellation?

The executable baseline provides diagnostic occurrence screens. The owner's year-end target requires cycle-aware valid-payment evidence, which is explicitly pending. An occurrence screen cannot establish that target.

In [ ]:
runs = challenge(YEAR, data_folder=data_root)
display(summary(runs))

for identifier, run in runs.items():
    if run["record"]["status"] != "computed":
        display(run["record"])
        raise RuntimeError("Resolve the recorded source or activity-binding issue before analysis.")
    display(Markdown("### " + identifier))
    display(run["specification"]["business_goal"])

## 2. Extraction: establish the evidence boundary

The public log records German agricultural direct-payment administration across several years. Actions occur within documents and can be performed manually or in batches. The source includes document identifiers, subprocess names, event identity and a success flag.

Activity labels are qualified as `doctype|subprocess|activity`. Generic labels such as `decide` or `begin payment` can describe different document workflows. Preserve raw labels and document identifiers when presenting witnesses. The source's binned amounts and area are not exact financial-balancing values.

Source fingerprints and dataset counts appear in the execution summary. The integrity record below checks numerical norm round-trip agreement, signed decomposition, case-index agreement and the count of events retained by the log object. It is not an independent certification of raw-source completeness or event-by-event retention. Criterion-level availability appears later. These numerical checks do not establish administrative and policy correctness.

In [ ]:
integrity_records = {}
for identifier, run in runs.items():
    integrity_records[identifier] = integrity(run)
    display(Markdown("### " + identifier))
    display(integrity_records[identifier])

## 3. Data processing: preserve whole-case context

Inspect funding `year`, `department`, inspection-selection fields, rejection status and parcel context across the whole application. Funding year is a source case attribute; it is distinct from the calendar year of an arbitrary event.

Department differences can reflect eligibility, parcel complexity, inspection selection, annual rule changes or documented workflow changes. Inspection-selected cases are not a random comparison cohort. The selection field's final observed value may be retrospective unless its availability time is established. Source amount and area bins cannot support precise savings estimates.

Read the profile of missing and mixed context before comparing groups. The case-start and last-observed month labels use the earliest and latest usable recorded timestamps after UTC normalisation. They can differ from business-local calendar months and do not verify the actual process start or finish. Verified completion needs a reviewed terminal-state definition. The observed span is the interval between these recorded endpoints; it does not establish processing time. Calendar selections must retain all events of each selected case.

In [ ]:
context_tables = {}
for identifier, run in runs.items():
    context_tables[identifier] = context_profiles(run, attributes=['year', 'department', 'number_parcels', 'selected_random', 'selected_risk', 'selected_manually', 'rejected', 'area'])
    display(Markdown("### " + identifier))
    display(context_tables[identifier])

### Turn goals into inspectable review criteria

The six occurrence criteria screen payment aborts, decision revocations, repeated initial decisions and editing starts, and payment starts in Change or Objection subprocesses. Their declared layers separate payment exceptions, decision revision, coordination and amendments.

Repeated amendment or objection work can be necessary and beneficial. A positive severity asks why the pattern occurred; it does not establish an administrative mistake or a failed entitlement decision.

Two stronger goals remain pending: a surviving valid initial payment start by funding-year end, and a decision-to-valid-payment interval. They require document and attempt identity, cancellation-aware matching, approved applicability and sufficient observation through the target date. Generic earliest-event pairing is inappropriate for those goals.

The table exposes the exact labels, scales, applicability, missing-data policy and stakeholder views used in the run. For a lag, the baseline uses the earliest usable start and the first usable end at or after it. It does not pair every cycle. Missing endpoints are unavailable evidence. Occurrence criteria count recorded event rows; they do not prove that repeated work was unnecessary. The Balanced view assigns equal total weight to each declared layer.

In [ ]:
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    display(norm_table(run))
    display(Markdown("**Questions awaiting additional semantics or policy evidence**"))
    display(run["specification"].get("pending_checks", []))

## 4. Mining and analysis: distinguish activity volume from case burden

Use qualified labels to compare aborts and revocations with initial, Change and Objection payment starts. A payment start followed by an abort can describe an unsuccessful attempt. Another start may establish a later valid attempt, but only when its document and cycle have been validated.

The first measure counts recorded occurrences. The second counts cases with at least one occurrence. Their units differ. Neither is an estimate of staff effort, avoidable cost or a causal explanation.

In [ ]:
activity_tables = {}
for identifier, run in runs.items():
    table = activity_summary(run, labels=['Payment application|Application|abort payment', 'Payment application|Change|abort payment', 'Payment application|Objection|abort payment', 'Payment application|Application|revoke decision', 'Payment application|Objection|revoke decision', 'Payment application|Application|decide', 'Payment application|Application|begin editing', 'Payment application|Change|begin payment', 'Payment application|Objection|begin payment'])
    activity_tables[identifier] = table
    display(Markdown("### " + identifier))
    display(table)
    if not table.empty:
        figure, axes = plt.subplots(1, 2, figsize=(14, 5), squeeze=False)
        table.set_index("activity")[["occurrences"]].plot.barh(
            ax=axes[0, 0], legend=False, color="C0"
        )
        table.set_index("activity")[["observed_cases"]].plot.barh(
            ax=axes[0, 1], legend=False, color="C1"
        )
        axes[0, 0].set_title("Recorded occurrences")
        axes[0, 1].set_title("Cases with an occurrence")
        axes[0, 0].set_xlabel("Event rows")
        axes[0, 1].set_xlabel("Whole cases")
        for axis in axes[0]:
            axis.xaxis.set_major_locator(MaxNLocator(5))
        figure.suptitle(identifier)
        figure.tight_layout()
        display(figure)
        plt.close(figure)

### Severity and evidence availability answer different questions

Occurrence criteria can be evaluated from the observed qualified labels, while source completeness remains a separate issue. Full availability for these screens does not mean the stronger valid-payment target is observable or implemented.

Read amendment and objection severities as review intensity. Their meaning differs from an avoidable abort or incorrect decision. Keep each criterion's description visible when explaining the overall score.

Inspect evaluated and in-scope counts beside mean severity. Evidence coverage is the fraction of in-scope cases that support a criterion evaluation. A low severity with little available evidence is inconclusive. A high severity under an illustrative scale still needs its process meaning checked.

WISE renormalises the usable criteria and layers. A high compliance score can therefore depend on partial evidence; it does not certify every declared requirement. The criterion table reports an unweighted fraction of cases with usable evidence. The case witness also reports native weighted evidence coverage for each view. Neither measure establishes source-log completeness.

In [ ]:
criterion_tables = {}
for identifier, run in runs.items():
    table = criterion_summary(run)
    criterion_tables[identifier] = table
    display(Markdown("### " + identifier))
    display(table)
    if not table.empty:
        axis = table.set_index("criterion_id")[["evidence_coverage", "mean_severity"]].plot.bar(
            figsize=(10, 4), ylim=(0, 1), rot=0
        )
        axis.set_ylabel("Fraction or mean severity")
        axis.set_title(identifier + ": evidence and review severity")
        axis.figure.tight_layout()
        display(axis.figure)
        plt.close(axis.figure)

### Choose a context for the investigation list

Start with funding year and department together. Then inspect year, department and inspection-selection fields as separate whole-case contexts. Do not attribute a department difference to performance without checking eligibility, inspection and annual workflow changes.

The Payment and Amendments views intentionally express different investigation interests. Their priorities should help choose evidence to inspect while preserving the distinction between necessary corrections and avoidable internal coordination.

Read the returned grouping and view specification beside each list. Priority combines case volume and a score gap relative to the population of this run. A zero relative priority does not establish an acceptable absolute result. Small-slice shrinkage is a declared sensitivity setting, not a fitted service target.

In [ ]:
investigation_lists = {}
for identifier, run in runs.items():
    investigation_lists[identifier] = {}
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=0)
        adjusted = priorities(run, view=view, gamma=20)
        investigation_lists[identifier][view] = backlog
        display(Markdown("### " + identifier + " / " + view))
        display(backlog.spec.to_dict())
        display(backlog.table.head(10))
        display(Markdown("**Same population with small-slice shrinkage**"))
        display(adjusted.table.head(10))
    for grouping in [['year', 'department'], 'year', 'department', 'selected_risk', 'selected_random']:
        backlog = priorities(run, by=grouping, gamma=0)
        display(Markdown("**Alternative whole-case grouping: " + str(grouping) + "**"))
        display(backlog.spec.to_dict())
        display(backlog.table.head(10))

### Read the overview together with its signed explanation

A department-year group can have a strong payment-exception contribution while a negative amendment contribution offsets part of its relative gap. Return to the qualified source activity before describing this as more or less successful administration. The graph of contributions does not identify a payment cycle or a causal dependency.

These figures use the evaluated cases and library-returned priorities and contributions. Positive layer contributions add to the relative penalty gap; negative contributions offset it. Preserve both signs when explaining a shortlisted group. The decomposition describes the score, not the cause of a process problem.

In [ ]:
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    show_figures(run)

### Return to the actual case

Inspect witnesses for aborted payment attempts and repeated editing. For each, follow `docid` and `docid_uuid`, subprocess, success and later events across the complete application. Check whether a revocation or new decision refers to the same document version.

The current baseline does not determine a surviving payment cycle. Record that judgement only after source and document review. Do not infer a deadline failure from an absent surviving payment when the observation horizon is insufficient.

The helper chooses a recorded witness for the requested criterion and returns the original source events, case context and evaluated criterion evidence. The full source trace remains available. Read rows with tied timestamps as unresolved ordering unless the source provides a stronger order. To inspect another case, pass its actual identifier as `case_id` to `case_evidence`.

In [ ]:
run = runs["bpic2018"]
for criterion in ("GR_01", "GR_04"):
    display(Markdown("### Source witness: " + criterion))
    show_evidence(case_evidence(run, criterion=criterion))

### Findings from the checked source baseline

The supplied application log contains **43,809 cases and 2,514,266 events**. The six occurrence screens are evaluable for every recorded case. This covers the occurrence questions; it does not establish the cancellation-aware year-end payment outcome.

Initial-application payment aborts appear in **26,625 cases**, with **32,936 recorded abort events**. A case can contain several attempts and later successful payment. Decision revocations appear in 20,621 cases. Before proposing a readiness checklist, inspect the document dependency or correction preceding selected aborts and establish whether it was preventable.

Under the baseline Balanced view, funding year 2016 and department `4e` lead the department-year list, with 4,531 cases and a Priority Index of 812.6643. Its positive signed gap concentrates in payment exceptions and decision revision; amendments provide a small negative offset. Payment-attempt readiness warrants investigation before a blanket reduction of amendment work.

Doubling occurrence saturation widths changes the leading group to 2015/`e7`, although top-ten positive-priority membership overlap remains 1.0. Shortlist agreement does not imply the same investigation order. Confirm the preferred review scales before allocating case-review capacity.

The source documents parcel-workflow changes across years. The higher 2017 baseline score is an observed contrast, not evidence that an intervention worked. Compare within-year departments and reviewed application classes before attributing differences to team performance. Inspection selection and retrospective derived attributes need separate treatment in prediction. [Source definitions and owner questions](https://ais.win.tue.nl/bpi/2018/challenge.html).

These findings refer to the source fingerprint and native norm in the execution record. Rerun after changing either. Review aborted attempts alongside later successes and legitimate controls, then trial a readiness check only if a repeated avoidable dependency is confirmed.


### Place the source witness on a timeline

This figure uses timestamps returned by the library for the complete recorded case selected by the first diagnostic criterion. Named lanes highlight norm activities; the grey lane retains the other recorded activities. Coincident points can hide multiplicity and establish no order. Undated events remain in the full source table. An apparent gap is a prompt to inspect the record, not evidence of staff waiting or a causal dependency.

In [ ]:
from bpic_analysis import plot_case_timeline
figure = plot_case_timeline(runs["bpic2018"])
display(figure)
plt.close(figure)

## 5. Evaluation: challenge the interpretation

First establish whether the Payment view selects a pattern of aborts rather than simply large applications with many document actions. Then ask whether inspection dependencies, reference-data gaps, a legitimate objection or avoidable handoffs explain the inspected attempts.

No current count criterion establishes timely valid payment or the effect of a proposed intervention. A later policy assessment must explicitly identify the eligible initial-payment cohort, surviving attempts and year-end observation coverage. That assessment should be evaluated separately from this diagnostic baseline.

The factors below scale both the lag threshold `delta` and saturation width `width`. For occurrence criteria they scale the saturation width `K` while keeping the integer occurrence tolerance `k` fixed. The helper rescores each declared scenario with WISE. These tests examine the chosen scales; they do not estimate an optimal policy. The view agreement table compares positive-priority shortlist membership under the declared views; it does not measure causal confidence or validate a business policy. Inspect a large disagreement by returning to its evidence.

In [ ]:
sensitivity_tables = {}
view_tables = {}
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    sensitivity_tables[identifier] = threshold_sensitivity(run, factors=(0.5, 1, 2))
    view_tables[identifier] = view_agreement(run, k=10)
    display(sensitivity_tables[identifier])
    display(view_tables[identifier])

## 6. Process improvement and support: define a feasible trial

| Confirmed mechanism required | Feasible change and owner | Pilot measure | Protected outcome |
|---|---|---|---|
| Required cross-document information is repeatedly missing when payment is started | Payment-operation lead trials a reviewed readiness checklist before one defined payment step | Confirmed avoidable aborts and time to readiness in the chosen workflow | Correct entitlement and all required inspections |
| An exception moves between teams without a clear next action | Department lead trials one exception owner and a visible dependency status | Unresolved dependency age and repeated internal routing | Legal review, independent controls and objection access |
| The same correction is entered repeatedly across linked documents | Application-service lead trials a reviewed correction record linked to source documents | Repeat entry of the same verified correction | Traceable document versions and correct decisions |

Begin with one department, funding year and workflow class. Validate payment-cycle interpretation before measuring the year-end outcome. Preserve authorised inspection and objection work even when it increases the number of events.

A change proceeds after the mechanism has been confirmed in source-record review and its owner has accepted the scope. Declare the baseline cohort, observation horizon, exceptions, outcome measures and stop conditions before the pilot. Record changes in case mix and evidence collection. A shorter recorded trace or lower diagnostic score alone does not establish a better process.

### Full analysis journal and review record

The execution journal below covers the run and its evidence checks. Add dated observations to `review_journal` as cases are discussed. Each entry should identify its supporting case or table, a plausible competing explanation, the reviewer decision and the next action. Empty fields remain unfilled work; they are not invented findings.

In [ ]:
execution_journal = journal(runs)
display(execution_journal)

review_journal = {
    "business_goal_and_population": "",
    "norms_approved_or_changed": "",
    "evidence_identifiers_reviewed": [],
    "observed_pattern": "",
    "competing_explanations": [],
    "confirmed_avoidable_mechanism": "",
    "decision_and_owner": "",
    "pilot_scope_and_comparison": "",
    "protected_outcomes_and_stop_conditions": [],
    "review_date": "",
    "next_action": "",
}
display(review_journal)

### Decision gates before reporting an improvement

- Does the evidence retain document, subprocess and application identity?
- Has each inspected payment attempt been linked to any later cancellation or revised attempt?
- Are eligibility, inspection selection and observation through the relevant deadline known?
- Has an avoidable readiness or coordination mechanism been confirmed before proposing a process change?
- Can the pilot measure valid payments and correct entitlement, rather than simply fewer recorded actions?

The portable output option saves the actual run records, tables and figures in `outputs`. Review case-level information before sharing saved outputs. The notebook itself starts with cleared outputs.

In [ ]:
if SAVE_OUTPUTS:
    output_folder = Path("outputs") / "bpic2018"
    save_outputs(runs, output_folder)
    display({"saved_outputs": output_folder.as_posix()})

## Source and method notes

- [BPI Challenge 2018 owner questions and data dictionary](https://ais.win.tue.nl/bpi/2018/challenge.html).
- [Public agricultural-grants event log](https://doi.org/10.4121/uuid:3301445f-95e8-4ff0-98a4-901f1f204972) and the supplied source metadata.
- [PM² methodology](https://vdaalst.com/publications/p824.pdf).

The declared norm and source fingerprints are recorded by the helper. WISE calculates constraint evaluations, scores, priorities and decompositions; notebook code selects and presents those results. Pending rules require the additional evidence stated in the norm specification.